# 通信用例实战

这一节用四个小用例练习组织通信测试：先改数据量，再换算子，然后增加参与者，
最后改变参与者所在的模拟服务器。每个用例把日志与配置存进自己的目录，方便比较和重跑。

**每个用例都是一个单元，点一次就跑完**（起会话 → 建通信域 → 跑算例 → Checker → 存记录）。

## 本节学习大纲

- 准备测试与记录目录
- 用例 01：两 rank ReduceScatter，改变数据量
- 用例 02：两 rank AllReduce，改变算子
- 用例 03：四 rank ReduceScatter，增加参与者
- 用例 04：跨模拟服务器，改变参与者分布
- 查看整组记录并导出

## 1. 准备测试与记录目录

四个用例共用一套环境变量，都来自 `env.sh`。本节新增的只有一个：`RECORD_ROOT`，
用来放四个用例各自的子目录。

| 用例 | 通信域 | MPI 进程数 | 算子 | 数据量 |
| --- | --- | ---: | --- | --- |
| 01 | 112：一台模拟服务器、两个 rank | 2 | ReduceScatter | 64、128、256 B |
| 02 | 112：一台模拟服务器、两个 rank | 2 | AllReduce | 64 B |
| 03 | 114：一台模拟服务器、四个 rank | 4 | ReduceScatter | 64 B |
| 04 | 122：两台模拟服务器、各两个 rank | 4 | ReduceScatter | 64 B |

数据量按每个 rank 的输入计算。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

export RECORD_ROOT="$VM_INSTALL/records/03_07_cases"
rm -rf "$RECORD_ROOT"
mkdir -p "$RECORD_ROOT"

echo "RECORD_ROOT = $RECORD_ROOT"
ls "$TEST_BIN/reduce_scatter_test" "$TEST_BIN/all_reduce_test"

末尾应列出两个算例。缺哪个说明包不完整，回到 03.03 重新解压。

每个用例结束后会留下这些文件：

| 文件 | 内容 |
| :--- | :--- |
| `test.log` | 算例的完整输出（结果表、告警） |
| `session.log` | 会话屏幕快照（含 Checker 的汇总） |
| `ranktable.json` | 本次通信域的 rank 与设备映射 |
| `topo.json` | 本次使用的服务器拓扑 |

## 2. 用例 01：两 rank ReduceScatter，改变数据量

`-b 64 -e 256 -f 2` 表示从 64 B 开始、每次翻倍、到 256 B 结束，
依次测试 64、128、256 B 三个数据量。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
CASE_DIR="$RECORD_ROOT/01_rs_sizes"
mkdir -p "$CASE_DIR"

# 1 start a fresh session
tmux kill-server 2>/dev/null
# kill-server 是异步的：等旧服务端真正退出，否则 new-session 会撞上"正在死的 server"
for i in $(seq 1 20); do
    tmux list-sessions >/dev/null 2>&1 || break
    sleep 0.2
done
for i in 1 2 3; do
    tmux new-session -d -s hvm -x 200 -y 50 "$PACK/run_hccl_vm.sh start $CLUSTER_YAML 2>&1 | tee /tmp/hvm_start.log" && break
    sleep 1
done
if ! tmux has-session -t hvm 2>/dev/null; then
    echo "!! session not started, start output below:"
    tail -20 /tmp/hvm_start.log 2>/dev/null || true
fi
for i in $(seq 1 40); do
    tmux capture-pane -pt hvm 2>/dev/null | grep -q listening && break
    sleep 1
done

# 2 build comm domain
tmux send-keys -t hvm "$VM mock-comm 112" Enter
sleep 8

# 3 run the case
tmux send-keys -t hvm "mpirun --oversubscribe -np 2 $TEST_BIN/reduce_scatter_test -b 64 -e 256 -f 2 -d int32 -o sum -w 0 -n 1 -c 1 > $CASE_DIR/test.log 2>&1" Enter
for i in $(seq 1 40); do
    grep -q "check_result" "$CASE_DIR/test.log" 2>/dev/null && break
    sleep 2
done

# 4 run checker
tmux send-keys -t hvm "$VM plugin run @checker" Enter
for i in $(seq 1 30); do
    tmux capture-pane -pt hvm | grep -q "CHECKER_RUN_SUMMARY" && break
    sleep 2
done

# 5 save
tmux capture-pane -pt hvm -S -200 > "$CASE_DIR/session.log"
cp "$VM_INSTALL/data/ranktable.json" "$CASE_DIR/" 2>/dev/null
cp "$VM_INSTALL/data/topo.json" "$CASE_DIR/" 2>/dev/null

echo "--- check_result ---"
grep -nE "check_result|^[[:space:]]*(64|128|256)[[:space:]]*\|" "$CASE_DIR/test.log"
echo "--- checker ---"
grep -a "CHECKER_RUN_SUMMARY" "$CASE_DIR/session.log" | tail -1 || true

三条结果行的 `check_result` 都应是 `failed`——**这是正常的**，模拟器不做数值计算（见 03.04 第 5 节）。

看两个规律：

- `avg_time` 三行都是 `1000.00 us`——模拟环境的固定值，不是真实耗时；
- `alg_bandwidth` 随数据量翻倍而翻倍（0.00006 → 0.00013 → 0.00026 GB/s），
  这是"数据量翻倍、耗时不变"的算术结果，同样不代表真实性能。

`checker` 那行应是 `All Success (Total Op: 6, ...)`：三个数据规模 × 每个规模 2 次校验调用。

每个 rank 的输出是输入的一半：

| 每 rank 输入 | int32 元素数 | 每 rank 输出 | int32 元素数 |
| --- | ---: | --- | ---: |
| 64 B | 16 | 32 B | 8 |
| 128 B | 32 | 64 B | 16 |
| 256 B | 64 | 128 B | 32 |

## 3. 用例 02：两 rank AllReduce，改变算子

仍是两个 rank，只把算例换成 `all_reduce_test`，数据量固定 64 B。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
CASE_DIR="$RECORD_ROOT/02_allreduce"
mkdir -p "$CASE_DIR"

# 1 start a fresh session
tmux kill-server 2>/dev/null
# kill-server 是异步的：等旧服务端真正退出，否则 new-session 会撞上"正在死的 server"
for i in $(seq 1 20); do
    tmux list-sessions >/dev/null 2>&1 || break
    sleep 0.2
done
for i in 1 2 3; do
    tmux new-session -d -s hvm -x 200 -y 50 "$PACK/run_hccl_vm.sh start $CLUSTER_YAML 2>&1 | tee /tmp/hvm_start.log" && break
    sleep 1
done
if ! tmux has-session -t hvm 2>/dev/null; then
    echo "!! session not started, start output below:"
    tail -20 /tmp/hvm_start.log 2>/dev/null || true
fi
for i in $(seq 1 40); do
    tmux capture-pane -pt hvm 2>/dev/null | grep -q listening && break
    sleep 1
done

# 2 build comm domain
tmux send-keys -t hvm "$VM mock-comm 112" Enter
sleep 8

# 3 run the case
tmux send-keys -t hvm "mpirun --oversubscribe -np 2 $TEST_BIN/all_reduce_test -b 64 -e 64 -d int32 -o sum -w 0 -n 1 -c 1 > $CASE_DIR/test.log 2>&1" Enter
for i in $(seq 1 40); do
    grep -q "check_result" "$CASE_DIR/test.log" 2>/dev/null && break
    sleep 2
done

# 4 run checker
tmux send-keys -t hvm "$VM plugin run @checker" Enter
for i in $(seq 1 30); do
    tmux capture-pane -pt hvm | grep -q "CHECKER_RUN_SUMMARY" && break
    sleep 2
done

# 5 save
tmux capture-pane -pt hvm -S -200 > "$CASE_DIR/session.log"
cp "$VM_INSTALL/data/ranktable.json" "$CASE_DIR/" 2>/dev/null
cp "$VM_INSTALL/data/topo.json" "$CASE_DIR/" 2>/dev/null

echo "--- check_result ---"
grep -nE "check_result|^[[:space:]]*(64)[[:space:]]*\|" "$CASE_DIR/test.log"
echo "--- checker ---"
grep -a "CHECKER_RUN_SUMMARY" "$CASE_DIR/session.log" | tail -1 || true

`checker` 应是 `All Success (Total Op: 2, ...)`：只有一个数据规模，每个规模 2 次校验调用。

**`Total Op` 由数据规模数决定，与 rank 数无关**——这条规律在下面两个用例里还会看到。

AllReduce 与 ReduceScatter 的数值差别：

- AllReduce 把各 rank 同一位置的数值相加，**每个 rank 都拿到完整结果**，输入输出同为 64 B；
- ReduceScatter 先求和再按 rank 均分，每个 rank 只拿自己那一块。

默认测试数据里，AllReduce 每个元素是 2，两 rank 相加得 4；
ReduceScatter 的输入按结果块初始化（前半 1、后半 2），两块分别得 2 和 4。

## 4. 用例 03：四 rank ReduceScatter，增加参与者

通信域改为 `114`（同一台模拟服务器上的四个 rank），MPI 进程数同步改成 4。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
CASE_DIR="$RECORD_ROOT/03_rs_four_ranks"
mkdir -p "$CASE_DIR"

# 1 start a fresh session
tmux kill-server 2>/dev/null
# kill-server 是异步的：等旧服务端真正退出，否则 new-session 会撞上"正在死的 server"
for i in $(seq 1 20); do
    tmux list-sessions >/dev/null 2>&1 || break
    sleep 0.2
done
for i in 1 2 3; do
    tmux new-session -d -s hvm -x 200 -y 50 "$PACK/run_hccl_vm.sh start $CLUSTER_YAML 2>&1 | tee /tmp/hvm_start.log" && break
    sleep 1
done
if ! tmux has-session -t hvm 2>/dev/null; then
    echo "!! session not started, start output below:"
    tail -20 /tmp/hvm_start.log 2>/dev/null || true
fi
for i in $(seq 1 40); do
    tmux capture-pane -pt hvm 2>/dev/null | grep -q listening && break
    sleep 1
done

# 2 build comm domain
tmux send-keys -t hvm "$VM mock-comm 114" Enter
sleep 8

# 3 run the case
tmux send-keys -t hvm "mpirun --oversubscribe -np 4 $TEST_BIN/reduce_scatter_test -b 64 -e 64 -d int32 -o sum -w 0 -n 1 -c 1 > $CASE_DIR/test.log 2>&1" Enter
for i in $(seq 1 40); do
    grep -q "check_result" "$CASE_DIR/test.log" 2>/dev/null && break
    sleep 2
done

# 4 run checker
tmux send-keys -t hvm "$VM plugin run @checker" Enter
for i in $(seq 1 30); do
    tmux capture-pane -pt hvm | grep -q "CHECKER_RUN_SUMMARY" && break
    sleep 2
done

# 5 save
tmux capture-pane -pt hvm -S -200 > "$CASE_DIR/session.log"
cp "$VM_INSTALL/data/ranktable.json" "$CASE_DIR/" 2>/dev/null
cp "$VM_INSTALL/data/topo.json" "$CASE_DIR/" 2>/dev/null

echo "--- check_result ---"
grep -nE "check_result|^[[:space:]]*(64)[[:space:]]*\|" "$CASE_DIR/test.log"
echo "--- checker ---"
grep -a "CHECKER_RUN_SUMMARY" "$CASE_DIR/session.log" | tail -1 || true

每个 rank 输入 64 B、16 个整数；分成四块后，每个 rank 输出 16 B、4 个整数。

这份算例把输入分成四块分别填 1、2、3、4，四个 rank 提供相同输入，求和后四块为 4、8、12、16：

```
Rank 0: input [1x4 | 2x4 | 3x4 | 4x4] -> output [4, 4, 4, 4]
Rank 1: input [1x4 | 2x4 | 3x4 | 4x4] -> output [8, 8, 8, 8]
Rank 2: input [1x4 | 2x4 | 3x4 | 4x4] -> output [12, 12, 12, 12]
Rank 3: input [1x4 | 2x4 | 3x4 | 4x4] -> output [16, 16, 16, 16]
```

`checker` 仍是 `Total Op: 2`——四个 rank 但只有一个数据规模。

## 5. 用例 04：跨模拟服务器，改变参与者分布

数据量、算子和四个进程都不变，通信域改成 `122`：
Server 0 取本地设备 0、1，Server 1 也取本地设备 0、1，共四个全局 rank。

变化发生在 **rank 的放置位置**和任务所用链路上——同一次集合通信从一台模拟服务器扩展到两台。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
CASE_DIR="$RECORD_ROOT/04_rs_cross_server"
mkdir -p "$CASE_DIR"

# 1 start a fresh session
tmux kill-server 2>/dev/null
# kill-server 是异步的：等旧服务端真正退出，否则 new-session 会撞上"正在死的 server"
for i in $(seq 1 20); do
    tmux list-sessions >/dev/null 2>&1 || break
    sleep 0.2
done
for i in 1 2 3; do
    tmux new-session -d -s hvm -x 200 -y 50 "$PACK/run_hccl_vm.sh start $CLUSTER_YAML 2>&1 | tee /tmp/hvm_start.log" && break
    sleep 1
done
if ! tmux has-session -t hvm 2>/dev/null; then
    echo "!! session not started, start output below:"
    tail -20 /tmp/hvm_start.log 2>/dev/null || true
fi
for i in $(seq 1 40); do
    tmux capture-pane -pt hvm 2>/dev/null | grep -q listening && break
    sleep 1
done

# 2 build comm domain
tmux send-keys -t hvm "$VM mock-comm 122" Enter
sleep 8

# 3 run the case
tmux send-keys -t hvm "mpirun --oversubscribe -np 4 $TEST_BIN/reduce_scatter_test -b 64 -e 64 -d int32 -o sum -w 0 -n 1 -c 1 > $CASE_DIR/test.log 2>&1" Enter
for i in $(seq 1 40); do
    grep -q "check_result" "$CASE_DIR/test.log" 2>/dev/null && break
    sleep 2
done

# 4 run checker
tmux send-keys -t hvm "$VM plugin run @checker" Enter
for i in $(seq 1 30); do
    tmux capture-pane -pt hvm | grep -q "CHECKER_RUN_SUMMARY" && break
    sleep 2
done

# 5 save
tmux capture-pane -pt hvm -S -200 > "$CASE_DIR/session.log"
cp "$VM_INSTALL/data/ranktable.json" "$CASE_DIR/" 2>/dev/null
cp "$VM_INSTALL/data/topo.json" "$CASE_DIR/" 2>/dev/null

echo "--- check_result ---"
grep -nE "check_result|^[[:space:]]*(64)[[:space:]]*\|" "$CASE_DIR/test.log"
echo "--- checker ---"
grep -a "CHECKER_RUN_SUMMARY" "$CASE_DIR/session.log" | tail -1 || true

结果数值与用例 03 相同（每个 rank 输入 64 B、输出 16 B），`checker` 仍是 `Total Op: 2`。

四个进程仍然都跑在**这一台实例**上，不需要第二台机器或 MPI hostfile——
模拟服务器和运行测试程序的电脑是两个层次。

> 想确认 rank 落在哪台模拟服务器，用 03.06 第 4 节的 `show_ranks`。

## 6. 查看整组记录并导出

四个用例跑完后，一次性汇总它们的结论。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

for case_dir in "$RECORD_ROOT"/*; do
  [ -d "$case_dir" ] || continue
  echo "=== $(basename "$case_dir") ==="
  grep -cE "check result failed|\[(error|fatal)\]" "$case_dir/test.log" 2>/dev/null | sed 's/^/  failures: /'
  grep -a "CHECKER_RUN_SUMMARY" "$case_dir/session.log" 2>/dev/null | tail -1 | sed 's/^/  /'
done

每个目录应满足四项：算例退出正常、结果行齐全、没有 `check result failed`、Checker 汇总非空且 `All Success`。

只看 root rank 打印的结果会漏掉其他 rank 的失败——所以要看 `test.log` 里的失败计数和 Checker 汇总。

最后把整组记录打包导出（实例关了就没了）：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

OUT="$HOME/hccl_vm_cases_$(date +%Y%m%d_%H%M%S).tar.gz"
tar czf "$OUT" -C "$(dirname "$RECORD_ROOT")" "$(basename "$RECORD_ROOT")"
ls -lh "$OUT"
echo "exported: $OUT"

在左侧文件列表里找到 `hccl_vm_cases_*.tar.gz`，**右键 → Download**。

## 7. 遇到问题先看这里

| 现象 | 原因 | 处理 |
| :--- | :--- | :--- |
| 某个用例的 `checker` 行是空的 | Checker 还没跑完就抓了屏幕 | 重跑该单元（会重建会话） |
| `Total Op` 比预期少 | 通信域和 `-np` 不匹配 | 检查该用例的 `mock-comm` 编号与 `-np` |
| 用例 01 只有 1 行结果 | `-f 2` 没生效或 `-e` 写错 | 确认参数是 `-b 64 -e 256 -f 2` |
| 所有用例都没有 `checker` 输出 | 会话被上一单元 `tmux kill-server` 关掉了 | 每个用例单元自己会重建会话，按顺序跑即可 |

[← 3.6 配置自己的通信测试](./03.06_configure_tests.ipynb) · [返回本章目录](./03.01_chapter_intro.ipynb) · [3.8 问题定位 →](./03.08_debug_and_verify.ipynb)